# Notebook 15: Makefiles and Build Configurations

Compiling C++ manually with long commands gets old fast. **Makefiles** automate the build process. Understanding compiler flags lets you control warnings, language standards, and optimisations.

This notebook is primarily a reference — most content is shown as shell/Makefile code blocks rather than executable C++ cells.

## The Compilation Pipeline

Compiling a C++ program happens in three phases:

```
  .cpp files
      │
      ▼  Preprocessor  (#include, #define, #ifdef ...)
  expanded .cpp
      │
      ▼  Compiler  (syntax check, code generation)
  .o object files  (machine code, unresolved external symbols)
      │
      ▼  Linker  (resolve symbols, combine .o files)
  executable binary
```

- **Preprocessing** — expands `#include` directives, replaces macros, processes `#ifdef` guards.
- **Compilation** — translates each `.cpp` file independently into a `.o` object file. This is where syntax errors are caught.
- **Linking** — combines all `.o` files and resolves references between them (e.g. `main.cpp` calls a function defined in `Shape.cpp`). Produces the final binary.

Each `.cpp` file is compiled independently. This means only changed files need recompiling — which is why Makefiles are useful.

## Compiling a Single File

The minimal command:

```bash
g++ -o hello hello.cpp
```

Explicitly set the language standard:

```bash
g++ -std=c++98 -o hello hello.cpp
```

At 42, you use `c++` (the school's compiler wrapper) rather than `g++` directly:

```bash
c++ -std=c++98 -Wall -Wextra -Werror -o hello hello.cpp
```

Multiple source files compiled and linked in one step:

```bash
c++ -std=c++98 -Wall -Wextra -Werror -o my_program main.cpp Shape.cpp Rectangle.cpp
```

## Common Compiler Flags

| Flag | Meaning |
|---|---|
| `-std=c++98` | Use the C++98 standard (required at 42) |
| `-std=c++11` | Use the C++11 standard |
| `-std=c++17` | Use the C++17 standard |
| `-Wall` | Enable all common warnings |
| `-Wextra` | Enable extra warnings beyond `-Wall` |
| `-Werror` | Treat every warning as a compile error |
| `-g` | Embed debug symbols (needed for `gdb`) |
| `-O0` | No optimisation (default) — best for debugging |
| `-O1` | Basic optimisations |
| `-O2` | Recommended for release builds |
| `-O3` | Aggressive optimisation (may change floating-point behaviour) |
| `-fsanitize=address` | Enable AddressSanitizer (detects out-of-bounds, use-after-free) |
| `-I./include` | Add `./include` to the header search path |
| `-o outputname` | Set the output file name |

> At 42, the canonical flags are `-Wall -Wextra -Werror -std=c++98`. Your code must compile cleanly with all three enabled.

## What is a Makefile?

A `Makefile` is a plain text file (named exactly `Makefile`) that describes how to build your project. Running `make` reads this file and executes the minimum set of commands needed to bring the project up to date.

**Key idea:** Make tracks file modification times. If `Shape.cpp` has not changed since `Shape.o` was last built, Make skips recompiling it. This saves time on large projects.

### Basic structure

```makefile
target: dependency1 dependency2
	recipe command
```

- **target** — the file to build (or a phony name like `all`, `clean`)
- **dependencies** — files the target depends on
- **recipe** — shell command(s) to build the target. **Must be indented with a real TAB character** (not spaces).

Make reads the first target in the file as the default goal (typically named `all`).

## A Basic Makefile

Here is the standard Makefile structure you will use at 42:

```makefile
NAME     = my_program
CXX      = c++
CXXFLAGS = -Wall -Wextra -Werror -std=c++98

SRCS = main.cpp foo.cpp bar.cpp
OBJS = $(SRCS:.cpp=.o)

all: $(NAME)

$(NAME): $(OBJS)
	$(CXX) $(CXXFLAGS) $(OBJS) -o $(NAME)

%.o: %.cpp
	$(CXX) $(CXXFLAGS) -c $< -o $@

clean:
	rm -f $(OBJS)

fclean: clean
	rm -f $(NAME)

re: fclean all

.PHONY: all clean fclean re
```

**What each part does:**

- `NAME`, `CXX`, `CXXFLAGS` — variables. Change the compiler or flags in one place.
- `SRCS` — list all your `.cpp` source files here.
- `OBJS = $(SRCS:.cpp=.o)` — substitution: replaces `.cpp` with `.o` in every entry.
- `all: $(NAME)` — default target; builds the binary.
- `$(NAME): $(OBJS)` — link all object files into the final binary.
- `%.o: %.cpp` — pattern rule: compile any `.cpp` → `.o`.
- `clean` — delete object files.
- `fclean` — delete object files **and** the binary.
- `re` — full rebuild from scratch.
- `.PHONY` — declares targets that are not real files.

## Makefile Variables

Variables make the Makefile maintainable — change a value once, it updates everywhere.

| Variable | Purpose |
|---|---|
| `CXX` | The C++ compiler (`c++`, `g++`, `clang++`) |
| `CXXFLAGS` | Compiler flags (warnings, standard, optimisation) |
| `NAME` | Output binary name |
| `SRCS` | List of `.cpp` source files |
| `OBJS` | List of `.o` object files (derived from `SRCS`) |

### Variable substitution

```makefile
SRCS = main.cpp Shape.cpp Rectangle.cpp
OBJS = $(SRCS:.cpp=.o)
# OBJS expands to: main.o Shape.o Rectangle.o
```

The syntax `$(VAR:from=to)` replaces every occurrence of `from` at the end of each word with `to`.

Reference a variable with `$(VARNAME)` or `${VARNAME}` — both work.

## Pattern Rules

A **pattern rule** uses `%` as a wildcard:

```makefile
%.o: %.cpp
	$(CXX) $(CXXFLAGS) -c $< -o $@
```

This means: **to build any `.o` file, compile the `.cpp` file with the same base name.**

| Special variable | Meaning |
|---|---|
| `$@` | The **target** of the current rule (e.g. `Shape.o`) |
| `$<` | The **first dependency** (e.g. `Shape.cpp`) |
| `$^` | **All dependencies** (space-separated) |

The `-c` flag tells the compiler to **compile only** (produce a `.o` file, do not link yet).

Example expansion for `Shape.o: Shape.cpp`:

```bash
c++ -Wall -Wextra -Werror -std=c++98 -c Shape.cpp -o Shape.o
#                                        ^^^          ^^^^
#                                        $<            $@
```

## Phony Targets

Targets like `clean`, `fclean`, `re`, `all` are **not actual files** — they are just named actions.

### The problem without `.PHONY`

If someone creates a file called `clean` in your project directory:

```bash
touch clean
make clean   # Make sees 'clean' exists and is up to date — does NOTHING!
```

### The fix: `.PHONY`

```makefile
.PHONY: all clean fclean re
```

This tells Make: **always run these targets, regardless of whether a file with that name exists.**

At 42, `.PHONY` is mandatory — the graders test it.

## Debug vs Release Configuration

Use Makefile conditionals to switch between a debug build (with sanitisers and debug symbols) and a release build (with optimisations):

```makefile
ifdef DEBUG
CXXFLAGS = -std=c++98 -Wall -Wextra -g -O0 -fsanitize=address
else
CXXFLAGS = -std=c++98 -Wall -Wextra -O2
endif
```

Build commands:

```bash
make              # Release build
make DEBUG=1      # Debug build with AddressSanitizer
```

### When to use each

| Build | Flags | Use when |
|---|---|---|
| Debug | `-g -O0 -fsanitize=address` | Development, hunting bugs |
| Release | `-O2` | Final submission, performance testing |

**Exercise 1 (written):**

Draw the directory structure for a project organised with separate `src/`, `include/`, and `obj/` directories:

```
my_project/
├── Makefile
├── include/
│   ├── Shape.hpp
│   └── Rectangle.hpp
├── src/
│   ├── main.cpp
│   ├── Shape.cpp
│   └── Rectangle.cpp
└── obj/
    (generated by make)
```

Write the Makefile for this project. Hints:
- `SRCS = $(wildcard src/*.cpp)` collects all `.cpp` files in `src/`
- `OBJS = $(SRCS:src/%.cpp=obj/%.o)` maps `src/X.cpp` → `obj/X.o`
- The pattern rule needs to create `obj/` if it does not exist: `mkdir -p obj`
- Add `-I./include` to `CXXFLAGS` so `#include "Shape.hpp"` works

## Multiple Source Files

A realistic project:

```
my_project/
├── Makefile
├── main.cpp
├── Shape.hpp
├── Shape.cpp
├── Rectangle.hpp
└── Rectangle.cpp
```

```makefile
NAME     = shapes
CXX      = c++
CXXFLAGS = -Wall -Wextra -Werror -std=c++98

SRCS = main.cpp Shape.cpp Rectangle.cpp
OBJS = $(SRCS:.cpp=.o)

all: $(NAME)

$(NAME): $(OBJS)
	$(CXX) $(CXXFLAGS) $(OBJS) -o $(NAME)

%.o: %.cpp
	$(CXX) $(CXXFLAGS) -c $< -o $@

clean:
	rm -f $(OBJS)

fclean: clean
	rm -f $(NAME)

re: fclean all

.PHONY: all clean fclean re
```

When you run `make`:
1. Make builds `main.o`, `Shape.o`, `Rectangle.o` in parallel if possible
2. Links them into `shapes`
3. If you then modify only `Rectangle.cpp` and run `make` again, only `Rectangle.o` and the final link step rerun

## Header Dependencies

**The problem:** If you change `Shape.hpp`, `Rectangle.cpp` needs to be recompiled (because it `#include`s `Shape.hpp`). But Make does not know this — it only sees `Rectangle.o: Rectangle.cpp` in our pattern rule.

**Manual solution:** Add explicit header dependencies:

```makefile
Rectangle.o: Rectangle.cpp Rectangle.hpp Shape.hpp
```

This gets tedious. The better solution is automatic dependency generation.

**Automatic solution with `-MMD -MP`:**

```makefile
CXXFLAGS = -Wall -Wextra -Werror -std=c++98 -MMD -MP
OBJS     = $(SRCS:.cpp=.o)
DEPS     = $(OBJS:.o=.d)

%.o: %.cpp
	$(CXX) $(CXXFLAGS) -c $< -o $@

-include $(DEPS)
```

- `-MMD` — while compiling, generate a `.d` file listing all headers `#include`d by this `.cpp`
- `-MP` — add empty phony rules for headers (prevents errors if a header is deleted)
- `-include $(DEPS)` — include all `.d` files (the `-` means: no error if they do not exist yet on the first build)

After one build, `Rectangle.d` might contain:

```makefile
Rectangle.o: Rectangle.cpp Rectangle.hpp Shape.hpp
```

Now Make knows to recompile `Rectangle.o` whenever any of those headers change.

## Final Exercise (written)

Write a complete `Makefile` for a project with these files:

```
main.cpp
Animal.hpp
Animal.cpp
Dog.hpp
Dog.cpp
```

Requirements:

- [ ] Binary named `zoo`
- [ ] C++98 standard
- [ ] All warnings enabled and treated as errors (`-Wall -Wextra -Werror`)
- [ ] Separate compile step (`.cpp` → `.o`) and link step
- [ ] `clean` target removes object files
- [ ] `fclean` target removes object files and the binary
- [ ] `re` target does a full rebuild
- [ ] `.PHONY` declared for all non-file targets
- [ ] **Bonus:** Add a `debug` target that rebuilds with `-g -O0 -fsanitize=address`

Write your Makefile in the cell below (as a comment or in a markdown cell).

In [ ]:
// Write your Makefile solution as a comment here:
//
// NAME     = ...
// CXX      = ...
// ...

## Modern C++ (C++11 and Beyond)

### Language Standard Flags

Simply change `-std=c++98` to select a newer standard:

| Flag | Standard | Key features |
|---|---|---|
| `-std=c++98` | C++98/03 | 42 requirement |
| `-std=c++11` | C++11 | `auto`, lambdas, move semantics, `nullptr`, range-for |
| `-std=c++14` | C++14 | Generic lambdas, `make_unique` |
| `-std=c++17` | C++17 | `std::optional`, structured bindings, `if constexpr` |
| `-std=c++20` | C++20 | Concepts, ranges, coroutines, `std::format` |

### CMake

For larger projects, [CMake](https://cmake.org/) is the modern standard. Instead of writing Make rules manually, you write a `CMakeLists.txt` describing your project and CMake generates the Makefile (or Ninja build files, Xcode project, etc.) for you.

```cmake
cmake_minimum_required(VERSION 3.10)
project(MyProject)
set(CMAKE_CXX_STANDARD 17)
add_executable(my_program main.cpp Shape.cpp Rectangle.cpp)
```

CMake is used by most real-world C++ projects. At 42, you use plain Makefiles.

### Compiler Explorer (godbolt.org)

[godbolt.org](https://godbolt.org) lets you type C++ code in the browser and immediately see the assembly output. Extremely useful for:
- Understanding what a compiler optimisation actually does
- Comparing `-O0` vs `-O2` output
- Seeing how virtual dispatch compiles to assembly

### `pkg-config`

When linking against installed libraries, `pkg-config` gives you the right flags:

```bash
c++ $(pkg-config --cflags --libs libpng) main.cpp -o my_program
```

In a Makefile:

```makefile
CXXFLAGS += $(shell pkg-config --cflags libpng)
LDFLAGS  += $(shell pkg-config --libs libpng)
```